# Sagas & the Outbox Pattern — the numbers behind Season 2, Part 4

Every figure in the post is recomputed here. Toy models are labelled as such; field figures are cited from the sources named in the post.

In [ ]:
import random
random.seed(42)

# --- 1. Compensation arithmetic: P(a saga needs an undo) = 1 - (1-p)^n ---
def p_needs_comp(p, n):
    return 1 - (1 - p) ** n

for n in (3, 5, 10):
    print(f"{n} steps, 1% per-step failure: {p_needs_comp(0.01, n)*100:.2f}% of sagas compensate")
assert abs(p_needs_comp(0.01, 5) - 0.0490099501) < 1e-9
# scale: 1M jobs/day, 10 steps, 1-in-1000 step failure (a1learner demo inputs)
print("1M jobs/day x 10 steps x 1/1000 failure ->",
      f"{int(1e6 * p_needs_comp(0.001, 10)):,} sagas/day need compensation")
assert int(1e6 * p_needs_comp(0.001, 10)) == 9955  # ~1 in 100 jobs

In [ ]:
# --- 2. Monte Carlo: unresolved sagas (a compensation fails -> human) ---
def run_sagas(trials=300_000, n=5, p=0.01, q=0.01):
    need_comp = unresolved = comp_actions = 0
    for _ in range(trials):
        comp = 0
        failed = False
        for k in range(n):
            if random.random() < p:
                failed = True
                comp = k  # k steps already committed need undoing
                break
        if not failed:
            continue
        need_comp += 1
        comp_actions += comp
        if any(random.random() < q for _ in range(comp)):
            unresolved += 1
    return need_comp / trials, unresolved / trials, comp_actions / max(need_comp, 1)

nc, un, avg_comp = run_sagas()
print(f"needs compensation: {nc*100:.2f}% | unresolved: {un*100:.4f}% (1 in {1/un:,.0f}) | avg comps per failed saga: {avg_comp:.2f}")
assert 0.045 < nc < 0.053
assert 0.0007 < un < 0.0013  # analytic 0.00096
# analytic cross-check
p, q, n = 0.01, 0.01, 5
analytic = sum((1-p)**(k-1)*p*(1-(1-q)**(k-1)) for k in range(1, n+1))
print(f"analytic P(unresolved) = {analytic:.6f} -> 1 in {1/analytic:,.0f}")
assert abs(analytic - 0.0009607416) < 1e-9

In [ ]:
# --- 3. Dual-write loss window: crash between commit and publish ---
# one crash per 30 days per service; window w between DB commit and broker publish
lam = 1 / (30 * 86400)  # crashes per second
for w_ms in (5, 100, 1000):
    lost = 10_000 * 86400 * lam * (w_ms / 1000)
    print(f"window {w_ms:>4} ms -> {lost:6.1f} lost/ghost events per day at 10k req/s")
assert abs(10_000 * 86400 * lam * 0.005 - 1.67) < 0.05
# outbox: the event commits atomically with the row -> 0 lost, by construction
print("outbox: 0 lost events (same transaction as the business row)")

In [ ]:
# --- 4. Relay latency: polling vs CDC, and the at-least-once duplicate rate ---
# polling: event waits Uniform(0, interval); CDC: ~0.3 s mean
for interval in (0.5, 1.0, 5.0):
    print(f"poll every {interval:.1f}s -> mean delay {interval/2:.2f}s, p99 ~{interval:.1f}s")
print("Debezium CDC (WAL tailing) -> mean ~0.30s, p99 ~0.80s (no polling load)")
# duplicates: relay crashes after publish, before marking progress -> redelivery
# if each publish has a 0.1% chance of a crash-before-ack, duplicates ~= that rate
trials, dup = 200_000, 0
for _ in range(trials):
    if random.random() < 0.001:
        dup += 1  # published twice
print(f"duplicate deliveries: {dup/trials*100:.3f}% of events (crash-before-ack 0.1%)")
# idempotent consumer collapses them: event-id set -> exactly-once effect
seen, applied, dup2 = set(), 0, 0
for i in range(trials):
    eid = i if random.random() >= 0.001 else i - 1  # redelivery reuses the id
    if eid in seen:
        dup2 += 1
        continue
    seen.add(eid)
    applied += 1
print(f"consumer applied {applied:,} of {trials:,} deliveries -> duplicates absorbed: {dup2:,}")
assert applied + dup2 == trials
assert abs(dup2 / trials - 0.001) < 0.0005

In [ ]:
# --- 5. Field figures (brahimbouine.com, 12 services, 8,000 tx/s) ---
# completion 99.7%, compensation triggered 0.3%, failed compensation 0.002%,
# avg duration 1.2s, p99 4.8s. Sanity: 0.3% of 8,200/s = compensations per second.
print(f"compensations/s at 8,200 tx/s x 0.3%: {8200*0.003:.1f}")
print(f"sagas/day at 8,200/s: {8200*86400:,}; compensating sagas/day: {int(8200*86400*0.003):,}")
print("2PC caused 3 outages in 2 months at the same shop -> sagas replaced it")